# UC Admissions Question Sprint — Fall 2025

This notebook shows the **Pandas work** for the 10 questions in the UC Admissions Data Challenge.

### Files used
- `dashboard_data.csv` — school × year × UC campus data
- `uc_freshman_admission_by_discipline.csv` — Fall 2025 admit rates by discipline
- `uc_admissions_summary_by_ethnicity.csv` — UC campus/systemwide race/ethnicity totals

> **Important:** `dashboard_data.csv` covers Bay Area public high schools, as described in its README. It is not an individual-student dataset.

For Question 1, the 2026 UC Accountability report states that Fall 2025 freshman applicants applied to an average of **4.5 UC campuses**.


In [ ]:
import pandas as pd
import numpy as np

dashboard = pd.read_csv("Data/dashboard_data.csv", low_memory=False)
discipline = pd.read_csv("Data/uc_freshman_admission_by_discipline.csv")
ethnicity = pd.read_csv("Data/uc_admissions_summary_by_ethnicity.csv")

print("dashboard:", dashboard.shape)
print("discipline:", discipline.shape)
print("ethnicity:", ethnicity.shape)


## 1. Average number of UC campuses applied to

**Answer: 4.50**

This value is reported directly by UC for Fall 2025, rather than being recoverable from the school-level CSV because the `Universitywide` rows count unduplicated students, not the number of campus applications.

In [ ]:
# Question 1
average_campuses = 4.50
round(average_campuses, 2)


## 2. Fall 2025 UCLA admit rate for applicants from California public high schools

Filter to:
- Fall 2025
- UCLA (`campus == "Los Angeles"`)
- public high schools

Then calculate **total admits / total applicants**. Do not average the school-level admit rates.

In [ ]:
q2 = dashboard[
    (dashboard["fall_term"] == 2025) &
    (dashboard["campus"] == "Los Angeles") &
    (dashboard["school_type"] == "High Schools (Public)")
].copy()

q2_applicants = q2["applicants"].sum()
q2_admits = q2["admits"].sum()
q2_rate = q2_admits / q2_applicants

print("Applicants:", q2_applicants)
print("Admits:", q2_admits)
print("Admit rate:", q2_rate)
print(f"Answer: {q2_rate:.2%}")


**Answer: 8.29%**

## 3. Which campus loses the most admit rate for Computer Science?

For each campus, compare the overall admit rate with the Computer Science admit rate:

`overall admit rate - CS admit rate`

A positive value means CS is harder to get into than the campus overall.

In [ ]:
q3 = discipline[
    (discipline["fall_term"] == 2025) &
    (discipline["broad_discipline"].isin(["All disciplines", "Computer Science"]))
].pivot(
    index="campus",
    columns="broad_discipline",
    values="admit_rate"
)

q3["CS_cost"] = q3["All disciplines"] - q3["Computer Science"]

q3.sort_values("CS_cost", ascending=False)


In [ ]:
q3_answer = q3["CS_cost"].idxmax()
q3_gap = q3.loc[q3_answer, "CS_cost"]

print("Campus:", q3_answer)
print(f"Difference: {q3_gap:.0%}")


**Answer: UC Davis — 25 percentage points**

## 4. Berkeley Computer Science admit-GPA IQR

The interquartile range is:

`75th percentile - 25th percentile`

In [ ]:
q4 = discipline[
    (discipline["fall_term"] == 2025) &
    (discipline["campus"] == "Berkeley") &
    (discipline["broad_discipline"] == "Computer Science")
].iloc[0]

iqr = q4["admit_gpa_p75"] - q4["admit_gpa_p25"]

print("P25:", q4["admit_gpa_p25"])
print("P75:", q4["admit_gpa_p75"])
print("IQR:", iqr)


**Answer: 0.09**

## 5. At how many UC campuses was the White freshman admit rate higher than the Hispanic/Latino(a) rate?

Use the UC race/ethnicity file. For each campus:

`admit rate = admits / applicants`

Then compare White with Hispanic/Latino(a).

In [ ]:
q5 = ethnicity[
    (ethnicity["fall_term"] == 2025) &
    (ethnicity["entrant_level"] == "freshman") &
    (ethnicity["ethnicity"].isin(["White", "Hispanic/Latino(a)"])) &
    (ethnicity["count_type"].isin(["App", "Adm"]))
].copy()

q5_table = q5.pivot_table(
    index=["campus", "ethnicity"],
    columns="count_type",
    values="n",
    aggfunc="first"
)

q5_table["admit_rate"] = q5_table["Adm"] / q5_table["App"]

q5_rates = q5_table["admit_rate"].unstack("ethnicity")
q5_rates["White_higher"] = (
    q5_rates["White"] > q5_rates["Hispanic/Latino(a)"]
)

q5_rates


In [ ]:
q5_answer = int(q5_rates["White_higher"].sum())
print("Number of campuses:", q5_answer)


**Answer: 9 campuses**

## 6. Systemwide: White or Hispanic/Latino(a)?

Restrict the same ethnicity table to `campus == "Systemwide"`.

In [ ]:
q6 = ethnicity[
    (ethnicity["fall_term"] == 2025) &
    (ethnicity["entrant_level"] == "freshman") &
    (ethnicity["campus"] == "Systemwide") &
    (ethnicity["ethnicity"].isin(["White", "Hispanic/Latino(a)"])) &
    (ethnicity["count_type"].isin(["App", "Adm"]))
].pivot_table(
    index="ethnicity",
    columns="count_type",
    values="n",
    aggfunc="first"
)

q6["admit_rate"] = q6["Adm"] / q6["App"]
q6


In [ ]:
q6_answer = q6["admit_rate"].idxmax()
print("Higher systemwide admit rate:", q6_answer)


**Answer: Hispanic/Latino(a)**

White = 68.68%; Hispanic/Latino(a) = 74.53%.

## 7. Bay Area Class of 2023 graduates enrolling at a California Community College

The dataset defines the Bay Area as these nine counties:

- Alameda
- Contra Costa
- Marin
- Napa
- San Francisco
- San Mateo
- Santa Clara
- Solano
- Sonoma

Use only the `Universitywide` row for each school so schools are not counted nine times. Then:

`CCC enrollment / graduates`

In [ ]:
bay_area_counties = [
    "Alameda", "Contra Costa", "Marin", "Napa",
    "San Francisco", "San Mateo", "Santa Clara",
    "Solano", "Sonoma"
]

q7 = dashboard[
    (dashboard["fall_term"] == 2023) &
    (dashboard["campus"] == "Universitywide") &
    (dashboard["county"].isin(bay_area_counties))
].copy()

graduates = q7["graduates"].sum()
ccc_enrollment = q7["enrolled_ccc"].sum()
q7_share = ccc_enrollment / graduates

print("Graduates:", graduates)
print("CCC enrollment:", ccc_enrollment)
print(f"Share: {q7_share:.2%}")


**Answer: 34.04%**

## 8. Mission San Jose High School: share of a-g completers who applied to at least one UC

The question explicitly gives the formula:

`Universitywide applicants / ag_completers`

In [ ]:
q8 = dashboard[
    (dashboard["fall_term"] == 2023) &
    (dashboard["campus"] == "Universitywide") &
    (dashboard["high_school"] == "MISSION SAN JOSE HIGH SCHOOL")
].iloc[0]

q8_share = q8["applicants"] / q8["ag_completers"]

print("Applicants:", q8["applicants"])
print("A-G completers:", q8["ag_completers"])
print(f"Share: {q8_share:.2%}")


**Answer: 99.06%**

## 9. Distinct California public high schools with at least one UC applicant in Fall 2025

The supplied dataset is specifically a **Bay Area public-high-school dataset**, despite the question's broader wording. Within the supplied data, filter to:

- Fall 2025
- `Universitywide`
- public high schools
- applicants > 0

Then count distinct school names.

In [ ]:
q9 = dashboard[
    (dashboard["fall_term"] == 2025) &
    (dashboard["campus"] == "Universitywide") &
    (dashboard["school_type"] == "High Schools (Public)") &
    (dashboard["applicants"] > 0)
].copy()

q9_answer = q9["high_school"].nunique()

print("Distinct public high schools with at least one applicant:", q9_answer)


**Answer from the supplied challenge dataset: 193**

## 10. Which listed school most outperforms its expected Berkeley admit rate?

The supplied `dashboard_data.csv` already contains:

- `expected_admit_rate`
- `admit_rate_residual`

The residual is the observed admit rate minus the expected admit rate after the model's controls. For 2022, `expected_admit_rate` is missing, so the usable controlled comparison is **2023–2025**.

The five schools named on the form are checked below.

In [ ]:
schools = [
    "HERCULES HIGH SCHOOL",
    "MISSION SENIOR HIGH SCHOOL",
    "MONTEREY TRAIL HIGH SCHOOL",
    "PHILLIP & SALA BURTON ACAD HS",
    "RANCHO SAN JUAN HIGH SCHOOL"
]

q10 = dashboard[
    (dashboard["campus"] == "Berkeley") &
    (dashboard["fall_term"].between(2022, 2025)) &
    (dashboard["high_school"].isin(schools))
].copy()

q10[[
    "fall_term", "high_school", "applicants", "admits",
    "admit_rate", "expected_admit_rate",
    "admit_rate_residual", "ag_completion_rate",
    "frpm_pct", "applicant_gpa", "cohort_students"
]].sort_values(["high_school", "fall_term"])


In [ ]:
q10_summary = (
    q10.groupby("high_school")
       .agg(
           mean_residual=("admit_rate_residual", "mean"),
           years_with_expected_rate=("admit_rate_residual", "count")
       )
       .sort_values("mean_residual", ascending=False)
)

q10_summary


In [ ]:
# Check whether any form-listed schools are absent from the supplied file.
missing_schools = sorted(set(schools) - set(q10["high_school"].unique()))
print("Missing from supplied CSV:", missing_schools)

print("\nTop observed controlled residual:")
print(q10_summary.head(1))


### Question 10 answer

**MISSION SENIOR HIGH SCHOOL**

Among the listed schools that are present in the supplied data, Mission Senior has the largest average positive residual (about **+0.250**, or +25.0 percentage points) across 2023–2025.

The supplied ZIP does **not** contain rows for Monterey Trail High School or Rancho San Juan High School, so this notebook does not invent values for them. Mission Senior is nevertheless the clear winner among the listed schools that are actually present in the challenge data.

# Final answers

1. **4.50**
2. **8.29%**
3. **UC Davis**
4. **0.09**
5. **9**
6. **Hispanic/Latino(a)**
7. **34.04%**
8. **99.06%**
9. **193**
10. **MISSION SENIOR HIGH SCHOOL**

### Sources
- Challenge data README: `Data/README.md`
